In [2]:
import hashlib
import os
import platform
import uuid

STUDENT_ID = 20244080223

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "exp02")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}{platform.machine()}{platform.system()}{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")

[Conda 环境] exp02
[学号] 20244080223
[机器指纹] df85dc8dfd4972ec
[RUN_ID] 20244080223-df85dc8dfd4972ec


字符串的乐趣

In [3]:
def is_palindrome(s):
    cleaned = [c.lower() for c in s if c.isalnum()]
    return cleaned == cleaned[::-1]

print(is_palindrome("taco cat"))      # True
print(is_palindrome("T A C O cat"))   # True
print(is_palindrome("hello"))         # False

True
True
False


In [4]:
def is_abecedarian(s):
    letters = [c.lower() for c in s if c != " "]
    return all(letters[i] <= letters[i+1] for i in range(len(letters)-1))

print(is_abecedarian("adder"))        # True
print(is_abecedarian("beet"))         # True
print(is_abecedarian("dog"))          # False
print(is_abecedarian("abcd efgh xyz"))# True

True
True
False
True


In [5]:
def remove_vowels(s):
    vowels = set("aeiouAEIOU")
    return "".join(c for c in s if c not in vowels)

print(remove_vowels("cat"))    # ct
print(remove_vowels("audio"))  # d
print(remove_vowels("aeiou"))  # ""

ct
d



列表的乐趣

In [6]:
def list_reverse(lst):
    if not isinstance(lst, list):
        raise TypeError("输入必须是列表")
    return lst[::-1]

print(list_reverse([1, 2, 3]))  # [3, 2, 1]

[3, 2, 1]


In [7]:
def is_sorted(seq):
    for i in range(len(seq) - 1):
        if seq[i] > seq[i+1]:
            return False
    return True

print(is_sorted([1, 2, 3]))    # True
print(is_sorted([1, 3, 2]))    # False

True
False


作为向量的元组

In [8]:
def _is_num(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)

def vec_scalar_mult(t, s):
    if not isinstance(t, tuple):
        raise TypeError("t 必须是元组")
    if not _is_num(s):
        raise TypeError("s 必须是整数或浮点数")
    for x in t:
        if not _is_num(x):
            raise TypeError("元组元素必须是整数或浮点数")
    return tuple(x * s for x in t)

print(vec_scalar_mult((1, 2, 3), 2))  # (2, 4, 6)

(2, 4, 6)


In [9]:
def vec_inner_product(x, y):
    if not isinstance(x, tuple) or not isinstance(y, tuple):
        raise TypeError("两个参数都必须是元组")
    if len(x) != len(y):
        raise ValueError("两个元组长度必须一致")
    return float(sum(a * b for a, b in zip(x, y)))

print(vec_inner_product((1, 2, 3), (4, 5, 6)))  # 32.0

32.0


In [10]:
def check_valid_mx(mx):
    if not isinstance(mx, tuple):
        return False
    if len(mx) == 0:
        return True
    if not all(isinstance(row, tuple) for row in mx):
        return False
    n = len(mx[0])
    if any(len(row) != n for row in mx):
        return False
    return all(_is_num(v) for row in mx for v in row)

In [11]:
def mx_vec_mult(mx, vec):
    if not check_valid_mx(mx):
        raise ValueError("无效矩阵")
    if not isinstance(vec, tuple):
        raise TypeError("向量必须是元组")
    if not all(_is_num(v) for v in vec):
        raise TypeError("向量元素必须是整数或浮点数")
    m = len(mx)
    n = len(mx[0]) if m > 0 else 0
    if len(vec) != n:
        raise ValueError("矩阵列数与向量维度不一致")
    return tuple(sum(mx[i][j] * vec[j] for j in range(n)) for i in range(m))

print(mx_vec_mult(((1, 2), (3, 4)), (1, 1)))  # (3, 7)

(3, 7)


更多向量的乐趣

In [12]:
class Vector:
    def __init__(self, dim, elems=None):
        if not isinstance(dim, int) or isinstance(dim, bool):
            raise TypeError("维度必须是整数")
        if dim < 0:
            raise ValueError("维度不能为负")
        if elems is None:
            elems = [0] * dim
        if not isinstance(elems, (list, tuple)):
            raise TypeError("元素必须是列表或元组")
        if len(elems) != dim:
            raise ValueError("维度与元素个数不一致")
        for v in elems:
            if not _is_num(v):
                raise TypeError("元素必须是整数或浮点数")
        self.dim = dim
        self.elems = tuple(elems)

    # 2. 维度与元素为实例属性：每个 Vector 对象有各自的值，
    #    若设为类属性会被所有实例共享，导致互相覆盖。

    def _check(self, other):
        if not isinstance(other, Vector):
            raise TypeError("必须是 Vector 对象")
        if self.dim != other.dim:
            raise ValueError("维度不同，无法比较/运算")

    def __eq__(self, other):
        self._check(other)
        return self.elems == other.elems

    def __lt__(self, other):
        self._check(other)
        return self.elems < other.elems

    def __le__(self, other):
        self._check(other)
        return self.elems <= other.elems

    def __gt__(self, other):
        self._check(other)
        return self.elems > other.elems

    def __ge__(self, other):
        self._check(other)
        return self.elems >= other.elems

    def dot(self, other):
        self._check(other)
        return float(sum(a * b for a, b in zip(self.elems, other.elems)))

    def __mul__(self, other):
        if _is_num(other):
            return Vector(self.dim, [x * other for x in self.elems])
        if isinstance(other, Vector):
            if self.dim != other.dim:
                raise ValueError("维度不同，无法相乘")
            return Vector(self.dim, [a * b for a, b in zip(self.elems, other.elems)])
        raise TypeError("只能与标量或 Vector 相乘")

    def __rmul__(self, other):
        return self.__mul__(other)

    def __repr__(self):
        return f"Vector({self.dim}, {self.elems})"

In [13]:
v = Vector(2, (2, 4))
w = Vector(2, (2, 5))
print(v < w)                 # True
print(v == Vector(2, [2, 4]))# True
print(v.dot(w))              # 24.0
print(2 * v, v * 2)          # Vector(2, (4, 8)) 各一次
print(v * w)                 # Vector(2, (4, 20))

True
True
24.0
Vector(2, (4, 8)) Vector(2, (4, 8))
Vector(2, (4, 20))
